# QT Cluster analysis

## Select Random?(Higher weight higher chance) Sample for clustering n=100

In [ ]:
import mdtraj as md
import numpy as np
import pandas as pd
import os

In [2]:
tag = "hcp_rdc_3j_theta=16"

# Extract the top 100 highest weighted frames

In [ ]:
BME_DIR = "bme_reweight"

w_rew = np.load(f"{BME_DIR}/crossvals/crossval_{tag}/weights_results.npy")[0]

w_rew_2 = np.column_stack((np.array(range(20100)), w_rew))
df = pd.DataFrame(w_rew_2)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_2.dat', index=False)

w_rew_3 = w_rew_2[np.argsort(w_rew_2[:,1])]
np.save(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_weights_frames.npy', w_rew_3)
df = pd.DataFrame(w_rew_3)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_weights_frames.dat', index=False)

w_rew_4 = np.argsort(w_rew_2[:,1])
np.save(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_frames.npy', w_rew_4)
df = pd.DataFrame(w_rew_4)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_frames.dat', index=False)

## Extract subensemble, by using random.choice, only run once!! without replacement

In [ ]:
BME_DIR = "bme_reweight"

weights = np.load(f"{BME_DIR}/crossvals/crossval_{tag}/weights_results.npy")[0]

traj_file = f"gcaa_simulations/concat_traj_nopbc.xtc"
top_file = f"gcaa_simulations/initial_nopbc_mdtraj.pdb"
pdb_file_out = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
n_frames_out = 100

frames_random = np.random.choice(np.arange(0, len(weights)), size=n_frames_out, replace=False, p=weights)# important modifier: replacement was set to False, instead of True

# Save Frame Clusters as PDB Files
np.save(f"qt_clustering_100/{tag}/subsampled_frames_random_noReplace.npy", frames_random)
traj = md.load(traj_file, top=top_file)
traj_subsampled = traj[frames_random]
traj_subsampled.save_pdb(pdb_file_out)


In [30]:
### weighting of the random subsample
summe = 0
for frame in frames_random:
    val = w_rew_2[frame][1]
    summe = summe + val
print("Weighting of the subensemble:", summe)

summe = 0
for frame in range(20100):
    val = w_rew_2[frame][1]
    summe = summe + val
print("Entire weighting of all frames:", summe)

Weighting of the subensemble: 0.23246746010710326
Entire weighting of all frames: 1.0000000000000004


# Quality Threshold Clustering

In [ ]:
traj_file = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
top_file = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
tag2 = "random_noReplace"

In [19]:
from lib import QT_fk_new as qtc
import barnaba as bb
import bz2
import pickle as cPickle

def save_bz2( outfile, results ):
    with bz2.BZ2File(outfile, 'w' ) as f:
        cPickle.dump(results, f, protocol = 4)

own_matrix_file = f'qt_clustering_100/{tag}/ownmatrix_{tag2}.pbz2'

trajj = md.load(traj_file, top=top_file)

N = trajj.n_frames
ermsd_matrix = np.zeros((N, N), dtype=np.float16)
for cluster_id in range(N):
    if cluster_id % 10 == 0:
        print(f'{cluster_id+10}/{N} frames processed.')
    ermsd_ = bb.ermsd_traj(trajj[cluster_id],trajj[cluster_id:],cutoff=2.4,residues_ref=[5,6,7,8,9,10],residues_target=[5,6,7,8,9,10])
    ermsd_matrix[cluster_id,cluster_id:] = ermsd_
tmp = np.array(ermsd_matrix[:,:]+ermsd_matrix[:,:].T, dtype=np.float32)
save_bz2(own_matrix_file, tmp)


10/100 frames processed.
20/100 frames processed.
30/100 frames processed.
40/100 frames processed.
50/100 frames processed.
60/100 frames processed.
70/100 frames processed.
80/100 frames processed.
90/100 frames processed.
100/100 frames processed.


In [20]:
ermsd_matrix = qtc.load_matrix(own_matrix_file)
cluster_arr = qtc.qt_cluster(ermsd_matrix, cutoff=0.9,minsize=5)
cut = "cutoff_0_9"
np.savetxt(f"qt_clustering_100/{tag}/QT_Clusters_{tag2}_{cut}_minsize5.txt", cluster_arr, fmt="%i")


Precalculated distance matrix provided.
Loading matrix...
Matrix Size 100
>>> Cluster # 1 found with 24 frames at center 6 <<<
>>> Cluster # 2 found with 12 frames at center 13 <<<
>>> Cluster # 3 found with 11 frames at center 89 <<<
>>> Cluster # 4 found with 10 frames at center 5 <<<
>>> Cluster # 5 found with 9 frames at center 10 <<<
>>> Cluster # 6 found with 6 frames at center 73 <<<
>>> Cluster # 7 found with 5 frames at center 23 <<<


### Load Clustering

In [21]:
cluster_array = np.loadtxt(f"qt_clustering_100/{tag}/QT_Clusters_{tag2}_{cut}_minsize5.txt")
cluster_dict = {}
for cluster_id in set(cluster_array):
    cluster_dict[int(cluster_id)] = np.where(cluster_array==cluster_id)[0]
print(cluster_dict)

{0: array([ 3,  4,  6,  8, 14, 16, 22, 28, 36, 38, 42, 44, 47, 49, 52, 57, 66,
       67, 70, 74, 77, 85, 87, 91], dtype=int64), 1: array([13, 25, 27, 31, 39, 46, 48, 56, 58, 64, 68, 80], dtype=int64), 2: array([ 1, 20, 21, 29, 40, 41, 75, 76, 84, 89, 90], dtype=int64), 3: array([ 0,  5, 12, 43, 50, 51, 63, 78, 86, 98], dtype=int64), 4: array([10, 17, 34, 55, 60, 65, 83, 95, 99], dtype=int64), 5: array([ 9, 11, 15, 69, 73, 97], dtype=int64), 6: array([23, 32, 61, 93, 94], dtype=int64), -1: array([ 2,  7, 18, 19, 24, 26, 30, 33, 35, 37, 45, 53, 54, 59, 62, 71, 72,
       79, 81, 82, 88, 92, 96], dtype=int64)}


### Save Clustered Frames in PDB Files

In [22]:
frames = np.load(f"qt_clustering_100/{tag}/subsampled_frames_{tag2}.npy")
print(frames)

[ 9051 11985 10016  7022   948  9856  6501 13921 19497 19175 17475 19231
 10453 16371  4966 19194  7542 17314 11731 14075 11753 11889  2502   740
 14072 16650  9570 16529  7044 11237  9771 16894  2428 13196 17455  8747
  7984 19573  2717 16752 15261 11960  8847 10389 13239 19156 16430 19082
 16318 19345 10458 10298  8817  1390  9807 17479 16836 19499 17079 10442
 17453  4319 10867 10288 16448 17188 13304  3500 16163  5323 13333 20084
 10858  5098  1810 15532 15687  7174  9473 14031 16497  1206  8530 17427
 11282  6775 10380 19337 11496 15172 15326  8410 11569  1740  6463 17473
  1533  3036 10331 17581]


In [23]:
traj_file = f"gcaa_simulations/concat_traj_nopbc.xtc"
top_file = f"gcaa_simulations/initial_nopbc_mdtraj.pdb"

traj = md.load(traj_file, top=top_file)
if not os.path.exists(f"qt_clustering_100/{tag}/frames_{tag2}_{cut}"):
    os.makedirs(f"qt_clustering_100/{tag}/frames_{tag2}_{cut}")
for cluster_id  in cluster_dict.keys():
   print(cluster_id)
   id = cluster_dict[cluster_id]
   print(frames[id])
   pdb_file_out = f'qt_clustering_100/{tag}/frames_{tag2}_{cut}/cluster_{cluster_id}.pdb'
   cluster = traj[frames[id]]
   cluster.save_pdb(pdb_file_out)


0
[ 7022   948  6501 19497  4966  7542  2502  7044  7984  2717  8847 13239
 19082 19345  8817 19499 13304  3500 13333  1810  7174  6775 19337  8410]
1
[16371 16650 16529 16894 16752 16430 16318 16836 17079 16448 16163 16497]
2
[11985 11753 11889 11237 15261 11960 15532 15687 11282 15172 15326]
3
[ 9051  9856 10453 10389 10458 10298 10288  9473 10380 10331]
4
[17475 17314 17455 17479 17453 17188 17427 17473 17581]
5
[19175 19231 19194  5323  5098  3036]
6
[ 740 2428 4319 1740 6463]
-1
[10016 13921 11731 14075 14072  9570  9771 13196  8747 19573 19156  1390
  9807 10442 10867 20084 10858 14031  1206  8530 11496 11569  1533]


In [24]:
frames_dict = {}
for cluster_id in set(cluster_array):
    id = cluster_dict[cluster_id]
    frames_dict[int(cluster_id)] = frames[id]
print(frames_dict)

{0: array([ 7022,   948,  6501, 19497,  4966,  7542,  2502,  7044,  7984,
        2717,  8847, 13239, 19082, 19345,  8817, 19499, 13304,  3500,
       13333,  1810,  7174,  6775, 19337,  8410]), 1: array([16371, 16650, 16529, 16894, 16752, 16430, 16318, 16836, 17079,
       16448, 16163, 16497]), 2: array([11985, 11753, 11889, 11237, 15261, 11960, 15532, 15687, 11282,
       15172, 15326]), 3: array([ 9051,  9856, 10453, 10389, 10458, 10298, 10288,  9473, 10380,
       10331]), 4: array([17475, 17314, 17455, 17479, 17453, 17188, 17427, 17473, 17581]), 5: array([19175, 19231, 19194,  5323,  5098,  3036]), 6: array([ 740, 2428, 4319, 1740, 6463]), -1: array([10016, 13921, 11731, 14075, 14072,  9570,  9771, 13196,  8747,
       19573, 19156,  1390,  9807, 10442, 10867, 20084, 10858, 14031,
        1206,  8530, 11496, 11569,  1533])}
